In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01_Environmnet_config

In [0]:
%run ../00-common/02_bronze_helpers

In [0]:
target_table = f"{catalog_name}.{gold_schema}.dim_drivers"
silver_table = f"{catalog_name}.{silver_schema}.drivers"
ref_nationality_table = f"{catalog_name}.{gold_schema}.ref_nationality_region"

In [0]:
drivers_df = spark.read.table(silver_table).filter(F.col("batch_id") == v_batch_id)
nationality_df = spark.read.table(ref_nationality_table)

In [0]:
dim_drivers_df = drivers_df.join(
    nationality_df, drivers_df.nationality == nationality_df.nationality, "left"
).select(
    drivers_df.driver_id,
    drivers_df.driver_name,
    drivers_df.date_of_birth,
    drivers_df.nationality,
    nationality_df.region.alias("nationality_region"),
)

In [0]:
write_to_gold (
    dim_drivers_df,
    target_table,
    "t.driver_id = s.driver_id",
    columns_to_update = [
        "driver_name",
        "date_of_birth",
        "nationality",
        "nationality_region"
    ]
)   

In [0]:
%sql
select
  *
from
  formula1_incr.gold.dim_drivers